# Companion analysis

What type of companions do WR stars have?

In [1]:
from posydon.utils.common_functions import convert_metallicity_to_string, orbital_separation_from_period
import matplotlib.pyplot as plt
import numpy as np


import pickle
from pathlib import Path

data_folder = Path('/Users/max/Documents/projects/WR_population/data/')

In [2]:
metallicity = 1
met_str = convert_metallicity_to_string(metallicity)
file_path = data_folder / f'WR_phases_Z{met_str}_Zsun.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)
    
S1_WR_data = data['S1_WR_data']
S2_WR_data = data['S2_WR_data']


In [3]:
time_array = []
weight_array = []
state_array = []
key_array = []
period_array = []
separation_array = []

companion_state_array = []


for key in S1_WR_data:
    data = S1_WR_data[key]['data']
    weights = S1_WR_data[key]['weights']
    
    time_array.extend(data['time'].values)
    weight_array.extend(weights.values)
    state_array.extend(data['state'].values)
    companion_state_array.extend(data['S2_state'].values)
    key_array.extend([key] * len(data))
    period_array.extend(data['orbital_period'].values)
    sep = orbital_separation_from_period(data['orbital_period'].values,
                                         data['S1_mass'].values,
                                         data['S2_mass'].values)
    
    separation_array.extend(sep)
    
for key in S2_WR_data:
    data = S2_WR_data[key]['data']
    weights = S2_WR_data[key]['weights']
    
    time_array.extend(data['time'].values)
    weight_array.extend(weights.values)
    companion_state_array.extend(data['S1_state'].values)
    state_array.extend(data['state'].values)
    key_array.extend([key] * len(data))
    period_array.extend(data['orbital_period'].values)
    sep = orbital_separation_from_period(data['orbital_period'].values,
                                         data['S1_mass'].values,
                                         data['S2_mass'].values)
    separation_array.extend(sep)
    
time_array = np.array(time_array)
weight_array = np.array(weight_array)
state_array = np.array(state_array)
key_array = np.array(key_array)
period_array = np.array(period_array)
separation_array = np.array(separation_array)
companion_state_array = np.array(companion_state_array)

In [4]:
np.unique(companion_state_array)

array(['BH', 'H-rich_Core_C_depleted', 'H-rich_Core_H_burning',
       'H-rich_Core_He_burning', 'H-rich_Core_He_depleted',
       'H-rich_Shell_H_burning', 'H-rich_non_burning', 'NS', 'WD',
       'massless_remnant', 'stripped_He_Core_C_depleted',
       'stripped_He_Core_He_burning', 'stripped_He_Core_He_depleted',
       'stripped_He_non_burning'], dtype='<U28')

In [ ]:
binary_mask = (state_array == 'detached') | (state_array == 'RLO1') | (state_array == 'contact') | (state_array == 'RLO2')

in_binary_companions = companion_state_array[binary_mask]

print(np.unique(in_binary_companions))

# BH
bh_mask = in_binary_companions == 'BH'
print('BH:', np.sum(weight_array[binary_mask][bh_mask]) / np.sum(weight_array[binary_mask]))
# NS
ns_mask = in_binary_companions == 'NS'
print('NS:', np.sum(weight_array[binary_mask][ns_mask]) / np.sum(weight_array[binary_mask]))
# WD
wd_mask = in_binary_companions == 'WD'
print('WD:', np.sum(weight_array[binary_mask][wd_mask]) / np.sum(weight_array[binary_mask]))
# stripped star
# 'stripped_He_Core_C_depleted', 'stripped_He_Core_He_burning',
# 'stripped_He_Core_He_depleted', 'stripped_He_non_burning'
ss_mask = np.flatnonzero(np.core.defchararray.find(in_binary_companions, 'stripped') != -1)
print('stripped:', np.sum(weight_array[binary_mask][ss_mask]) / np.sum(weight_array[binary_mask]))

ms_mask = in_binary_companions == 'H-rich_Core_H_burning'
print('MS:', np.sum(weight_array[binary_mask][ms_mask]) / np.sum(weight_array[binary_mask]))

# Hertzsprung gap
hg_mask = in_binary_companions == 'H-rich_Shell_H_burning'
print('HG:', np.sum(weight_array[binary_mask][hg_mask]) / np.sum(weight_array[binary_mask]))
# giants
g_mask = (in_binary_companions == 'H-rich_Core_He_burning') | (in_binary_companions == 'H-rich_Shell_He_burning') | (in_binary_companions == 'H-rich_He_depleted')
print('giants:', np.sum(weight_array[binary_mask][g_mask]) / np.sum(weight_array[binary_mask]))

# verification that this is very small:
final = (in_binary_companions == 'H-rich_Core_C_depleted')
print('final:', np.sum(weight_array[binary_mask][final]) / np.sum(weight_array[binary_mask]))

['BH' 'H-rich_Core_C_depleted' 'H-rich_Core_H_burning'
 'H-rich_Core_He_burning' 'H-rich_Core_He_depleted'
 'H-rich_Shell_H_burning' 'H-rich_non_burning' 'NS' 'WD'
 'stripped_He_Core_C_depleted' 'stripped_He_Core_He_burning'
 'stripped_He_Core_He_depleted' 'stripped_He_non_burning']
BH: 0.15825690995327688
NS: 0.00013102742644428324
WD: 2.4389018121201337e-06
stripped: 0.05364215012308521
MS: 0.7612496371280207
HG: 0.007323838576961742
giants: 0.018763194864828473
final: 1.0802514258131075e-07
